# Is the Track A score double-weighting anything?

The score the final 15 were picked by is one line of `examples/track_a_chem_rank.py::rank_weighted`:

$$\mathrm{score} = z(\mathrm{ipsae\_max}) + W \cdot \mathrm{chem\_z}, \qquad \mathrm{chem\_z} = z(\mathrm{contact}) + z(\mathrm{coulomb}), \qquad W = 0.15$$

so its only independent inputs are **ipsae_max, contact and coulomb** (`SCORE_INPUTS`). The other columns of
`ranking.csv` - `ipsae_min`, `iptm`, `in_band`, `n_on_site`, `para_net`, `chem` - are reported and never enter the
number.

Two inputs that measure the same thing would weight that thing twice against the rest, which is why `ipsae_min` is
kept out: it is close to a multiple of `ipsae_max`. This notebook checks every audited column for that pattern on the
committed ranking, and prints what each input is really worth once the inputs correlate.

Pure stdlib, no pandas: the functions come from the ranking script itself, so the notebook and the pipeline check are
the same code.

## Parameters

In [ ]:
RANKING = "../data/track_a_final/ranking.csv"   # what `rank --chem-weight 0.15` wrote, one row per candidate
CHEM_WEIGHT = 0.15                              # the weight that ranking was made with
THRESHOLD = 0.9                                 # flag a pair at or above this in |r| or |rho|

## Setup

In [ ]:
import csv
import importlib.util
import math
import statistics
import sys
from pathlib import Path

SCRIPT = Path("../examples/track_a_chem_rank.py").resolve()
_spec = importlib.util.spec_from_file_location("track_a_chem_rank", SCRIPT)
chem = importlib.util.module_from_spec(_spec)
sys.modules["track_a_chem_rank"] = chem
_spec.loader.exec_module(chem)

with open(RANKING, newline="") as handle:
    rows = list(csv.DictReader(handle))

print(f"{len(rows)} candidates from {RANKING}")
print("score inputs:", ", ".join(chem.SCORE_INPUTS))
print("audited:     ", ", ".join(chem.AUDIT_COLUMNS))
print("nominal weights:", chem.score_weights(CHEM_WEIGHT))

## 1. ipsae_min against ipsae_max

The regression that got `ipsae_min` excluded: slope, the SD of what is left over, and whether the two columns order
the candidates alike. A residual SD far below `ipsae_min`'s own SD means the column carries almost nothing
`ipsae_max` does not already carry.

In [ ]:
pairs = chem._both(rows, "ipsae_max", "ipsae_min")
x = [a for a, _ in pairs]
y = [b for _, b in pairs]
mx, my = statistics.mean(x), statistics.mean(y)
slope = sum((a - mx) * (b - my) for a, b in pairs) / sum((a - mx) ** 2 for a in x)
intercept = my - slope * mx
residuals = [b - (intercept + slope * a) for a, b in pairs]
r = chem.pearson(pairs)

print(f"n                 {len(pairs)}")
print(f"ipsae_min = {intercept:+.4f} + {slope:.4f} * ipsae_max")
print(f"SD(ipsae_min)     {statistics.pstdev(y):.4f}")
print(f"SD(residual)      {statistics.pstdev(residuals):.4f}")
print(f"variance explained {1 - statistics.pvariance(residuals) / statistics.pvariance(y):.3%}")
print(f"pearson r         {r:+.4f}")
print(f"spearman rho      {chem.spearman(pairs):+.4f}")

## 2. The correlation matrix over every audited column

Pearson first, then Spearman - the one that says whether two columns produce the same ordering. Cells are
pairwise-complete (a candidate with no chemistry still counts in the ipSAE pairs); `.` means the pair had fewer than
three usable rows or a constant column, `*` means it reached the threshold.

In [ ]:
for method in ("pearson", "spearman"):
    print(f"--- {method}\n")
    print(chem.format_correlation_matrix(rows, chem.AUDIT_COLUMNS, method, THRESHOLD), end="\n\n")

In [ ]:
# every audited pair, worst of |r| and |rho| first, so the matrix above can be read as a list
pear = chem.correlation_matrix(rows, chem.AUDIT_COLUMNS, "pearson")
spear = chem.correlation_matrix(rows, chem.AUDIT_COLUMNS, "spearman")


def worst(pair):
    values = [abs(v) for v in (pear.get(pair, math.nan), spear.get(pair, math.nan)) if not math.isnan(v)]
    return max(values) if values else -1.0


print(f"{'pair':<28}{'r':>8}{'rho':>8}{'n':>5}  note")
for pair in sorted(pear, key=worst, reverse=True):
    a, b = pair
    parents = chem.DERIVED_FROM.get(a, ()) + chem.DERIVED_FROM.get(b, ())
    note = []
    if worst(pair) >= THRESHOLD:
        note.append("OVER")
    if a in parents or b in parents:
        note.append("derived")
    if a in chem.SCORE_INPUTS and b in chem.SCORE_INPUTS:
        note.append("both in score")
    print(f"{a + ' / ' + b:<28}{pear[pair]:+8.3f}{spear.get(pair, math.nan):+8.3f}"
          f"{len(chem._both(rows, a, b)):5d}  {', '.join(note)}")

## 3. What the pipeline check says

`check_score_correlations` is what `rank` now runs on every ranking it writes. Over `SCORE_INPUTS` it is the gate; over
`AUDIT_COLUMNS` it is the wider report, where a pair marked *derived* (`chem`, `chem_z` and `in_band` are functions of
other columns) is expected and is not by itself a reason to drop anything.

In [ ]:
print("--- gate: the three columns the score depends on")
gate = chem.check_score_correlations(rows, THRESHOLD, chem.SCORE_INPUTS, stream=sys.stdout)
print("clean" if not gate else f"{len(gate)} pair(s) flagged")

print("\n--- wider report: every audited column")
wide = chem.check_score_correlations(rows, THRESHOLD, chem.AUDIT_COLUMNS, stream=sys.stdout)
print("clean" if not wide else f"{len(wide)} pair(s) flagged")

## 4. Effective weights, before and after a drop

A nominal weight is not what an input is worth: `score = sum_i w_i z_i`, so `Var(score) = sum_ij w_i w_j r_ij` and the
share input *i* carries is `w_i * sum_j w_j r_ij / Var`. Two correlated inputs lend each other weight. The cell prints
today's shares, then the shares if each input were removed - the "before and after" for any drop under discussion.

In [ ]:
before = chem.effective_weights(rows, CHEM_WEIGHT)
print("as scored:")
for name, share in before.items():
    print(f"  {name:<12}{share:8.1%}   nominal {chem.score_weights(CHEM_WEIGHT)[name]:g}")

for column in chem.SCORE_INPUTS:
    after = chem.effective_weights(rows, CHEM_WEIGHT, drop=(column,))
    print(f"\nwithout {column}:")
    for name, share in after.items():
        print(f"  {name:<12}{share:8.1%}   was {before[name]:.1%}")

print("\nipsae_min, iptm, in_band, n_on_site, para_net and chem carry 0% before and after: they are not in the score.")

## 5. Would a drop change the picks?

The only drop that can move a candidate is a drop of a scored column. This re-scores the committed ranking with one
input removed and reports how far the order moves, so a recommendation can be weighed against what it costs. It
re-ranks the written `score` columns rather than refolding anything.

In [ ]:
def rescore(records, weights):
    """Rank the written ranking.csv again under other weights on the same z-scores. A column with no value for a
    candidate contributes 0, as chem_z does in the pipeline."""
    zs = {}
    for column in weights:
        values = {r["name"]: chem._numeric(r.get(column)) for r in records}
        usable = {n: v for n, v in values.items() if not math.isnan(v)}
        mean, sd = statistics.mean(usable.values()), statistics.pstdev(usable.values())
        zs[column] = {n: (v - mean) / (sd if sd else 1.0) for n, v in usable.items()}
    scored = {r["name"]: sum(w * zs[c].get(r["name"], 0.0) for c, w in weights.items()) for r in records}
    return [n for n, _ in sorted(scored.items(), key=lambda kv: -kv[1])]


baseline = rescore(rows, chem.score_weights(CHEM_WEIGHT))
print(f"top 15 as scored: {baseline[:15]}")
for column in chem.SCORE_INPUTS:
    weights = {c: w for c, w in chem.score_weights(CHEM_WEIGHT).items() if c != column}
    order = rescore(rows, weights)
    moved = sum(1 for n in baseline[:15] if n not in order[:15])
    rho = chem.spearman([(float(baseline.index(n)), float(order.index(n))) for n in baseline])
    print(f"without {column:<12} rho(order) {rho:+.4f}   {moved} of the top 15 replaced   new top 15: {order[:15]}")

## How to read this

- A pair over the threshold that is **both in score** is the bug: one quantity weighted twice. Drop the member that is
  the derived or noisier one, keep the one the pipeline and the challenge are stated in terms of.
- A pair over the threshold where one column is **derived** from the other (`chem_z` from `contact` and `coulomb`,
  `in_band` from `ipsae_max`) is arithmetic, not a finding.
- A pair over the threshold where **neither is in the score** costs nothing: both columns are reported only. It is
  still worth knowing, because it says the two columns are not independent evidence when reading the table by eye.
- The sample is 115 candidates from one predictor at three seeds. Seed noise is large (see
  `docs/track_a_final_pipeline.md`), so read a correlation of 0.95 as "these are the same column" and one of 0.6 as
  "related, and the number itself is soft".